# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane:** Lane 2 — Refresh / Content Opportunity Scoring

**ML task type:** Supervised learning — specifically **binary classification used for ranking/scoring**.

We are not just predicting a rigid 0 or 1. The model outputs a **probability score** for each page, and those probabilities are sorted into a ranked review queue. This matches the editorial decision from Week 1: *which pages should a human reviewer look at first?*

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target/proxy:** `is_declining_label`, derived from `trend_direction == "down"`.

Since we do not have future-window data in the starter set yet, this is a **proxy label** — not an ideal capstone target, but adequate for learning the workflow. The model will predict the probability of a page falling into the declining bucket based on its historical 90-day **observable signals** (impressions, clicks, position, sessions, engagement, age, freshness, and related tiers).

**Honest caveat:** This label is computed from the current impression window, not from a later observed outcome. A stronger capstone will use prior-window features → future-window labels from the warehouse release.

**Leakage guard:** `trend_direction` and `trend_pct` define the label and must **never** be used as model features.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary metric: Precision@50**

Top-K metrics match the real-world editorial workflow. Of the top 50 pages the model flags for review, how many actually match the target criteria? If a team can act on ~50 candidates, precision@50 is the metric that matters most.

**Secondary health metrics:**

- **ROC AUC** — overall discrimination across all thresholds
- **Average Precision** — ranking quality across the full score distribution

**What "good" looks like (defined before training):**

The starter pipeline's verified benchmarks set the bar: the transparent rule baseline achieves **Precision@50 = 0.240** (~12 of 50 correct), while the random forest reaches **Precision@50 = 0.740** (~37 of 50 correct). A learned model should beat the baseline on this metric.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
import os
from pathlib import Path

import pandas as pd

# Resolve repo root from work/notebooks/
if not Path("data/raw/content_refresh_anonymized.csv").exists():
    if Path("../../data/raw/content_refresh_anonymized.csv").exists():
        os.chdir("../..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"Rows before dedupe: {len(df):,}")
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)
print(f"Rows after dedupe: {len(df):,}")
print(f"Unit of analysis: one row = one unique content item (content_id)\n")

df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(f"Declining label rate: {df['is_declining_label'].mean():.1%}\n")

preview_cols = [
    "content_id",
    "client_id",
    "impressions_90d",
    "content_age_days",
    "avg_position",
    "sessions_90d",
    "is_declining_label",
]
df[preview_cols].head()

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Fixed rules create arbitrary cliffs and miss nuance. A hand-written rule like `impressions > 500 AND age > 180 AND trend == down` treats every page as pass or fail — a page with 499 impressions is ignored entirely, even if its position, engagement, and session patterns suggest real risk.

Rules also fail to capture **non-linear interactions** between variables. A page's risk depends on how position, volume, and age combine — not on any single threshold in isolation.

ML learns these complex patterns from observable signals and outputs a **continuous probability score**. Editors receive a smoothly ranked queue prioritized by actual risk, not rigid cutoffs — which is exactly what Lane 2's review queue with reason codes is designed to deliver.

## Self-check

**ML-03 assignment requirements:**

- [x] **ML task type** stated — supervised binary classification used for ranking/scoring
- [x] **Target/proxy** defined — `is_declining_label` from `trend_direction == "down"`, with proxy caveat
- [x] **Success metric** defended — Precision@50 primary; ROC AUC and Average Precision secondary
- [x] **Unit of analysis** shown in a real dataframe — deduped by `content_id`, `head()` with identifiers, features, and label
- [x] **ML vs. fixed-rule explanation** included — arbitrary cliffs vs. continuous probability ranking

**Submission checklist:**

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.